# 19_final_training_data — 최종 학습데이터 (확장판: 6지문 + 2D + 3D)

**한 줄 요약:** 모델에 줄 **최종 표**를 만든다. 한 분자당 → SMILES + **6종 fingerprint** + **WEKA가 고른 2D descriptor** + **3D descriptor** + potency.
**이전과 차이:** 지문 1종(ECFP4)→**6종**, descriptor에 **3D**까지 추가.
**열 블록 구분(접두사):** `ecfp4_`,`rdkit_`,`atompair_`,`topotorsion_`,`maccs_`,`avalon_`(지문), `d3_`(3D). 나머지는 2D descriptor.
**큰 흐름:** ① 준비 → ② 재료 로드 → ③ 6지문 계산 → ④ 조립·저장

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기
6종 지문(Avalon 포함) 생성 도구를 가져온다.

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')  # data/ 폴더를 찾을 때까지 상위로 (하위 폴더에서 열어도 동작)
print('작업 폴더:', os.getcwd())
import numpy as np
import pandas as pd
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from rdkit.Avalon import pyAvalonTools
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
NB = 1024

🔎 **코드 뜯어보기 (준비)** *(chdir·import는 01, Avalon은 04에서 설명)*
- `NB=1024`=지문 길이. 6종을 위해 Morgan/RDKit/AtomPair/TopoTorsion 생성기 + MACCS/Avalon 함수를 쓴다.

### 셀 1 — 재료 3개 로드
(1) 2D descriptor 전체표(+potency), (2) WEKA가 고른 2D descriptor 이름, (3) 09b의 3D descriptor.

In [ ]:
# 재료 3개: (1) 2D descriptor 전체(+potency) (2) WEKA 선택 목록 (3) 3D descriptor
FULL2D = "data/HSD17B13_1to1_descriptors.xlsx"          # canonical_smiles+potency+217 2D desc
FILT   = "data/HSD17B13_1to1_descriptors_weka_filtered.csv"
D3     = "data/HSD17B13_1to1_3d_descriptors.csv"        # canonical_smiles + d3_*

base = pd.read_excel(FULL2D)                             # 기준 표(행 = 학습 분자)
sel2d = [c for c in pd.read_csv(FILT, nrows=0).columns if c.lower() != "potency"]  # 선택된 2D desc 이름
d3 = pd.read_csv(D3)                                     # 3D descriptor
print("기준 분자:", len(base), "| 선택 2D desc:", len(sel2d), "| 3D desc:", d3.shape[1]-1, "종")

🔎 **코드 뜯어보기 (셀 1)**
- `pd.read_excel(FULL2D)` : 18에서 만든 2D descriptor 표(분자별 potency 포함)를 **기준 표(base)** 로.
- `pd.read_csv(FILT, nrows=0).columns` : 헤더만 읽어 **선택된 2D descriptor 이름**. `pd.read_csv(D3)`=3D descriptor 표.

### 셀 2 — 6종 fingerprint 계산
학습 분자마다 6가지 지문을 계산해 종류별 표로 만든다(열 이름에 접두사).

In [ ]:
# 6종 fingerprint 계산 (학습 분자에 대해)
gens = {
    "ecfp4":       rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=NB),
    "rdkit":       rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=NB),
    "atompair":    rdFingerprintGenerator.GetAtomPairGenerator(fpSize=NB),
    "topotorsion": rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=NB),
}
def bits(fp, n):
    a = np.zeros((n,), dtype=np.int8); DataStructs.ConvertToNumpyArray(fp, a); return a

blocks = {k: [] for k in ["ecfp4", "rdkit", "atompair", "topotorsion", "maccs", "avalon"]}
for smi in base["canonical_smiles"]:
    m = Chem.MolFromSmiles(str(smi))
    for k in ("ecfp4", "rdkit", "atompair", "topotorsion"):
        blocks[k].append(gens[k].GetFingerprintAsNumPy(m))
    blocks["maccs"].append(bits(MACCSkeys.GenMACCSKeys(m), 167))
    blocks["avalon"].append(bits(pyAvalonTools.GetAvalonFP(m, NB), NB))

fp_dfs = []
for k, mat in blocks.items():
    X = np.vstack(mat)
    fp_dfs.append(pd.DataFrame(X, columns=[f"{k}_{j:04d}" for j in range(X.shape[1])]))
print("fingerprint 6종 계산 완료:", {k: len(v[0]) for k, v in blocks.items()})

🔎 **코드 뜯어보기 (셀 2)**
- `gens = {"ecfp4": ..., ...}` : 생성기 4종을 딕셔너리에. `def bits(fp, n):`=MACCS/Avalon 지문을 0/1 배열로.
- `blocks = {k: [] for k in [...]}` : 6종 지문을 담을 리스트들. 분자마다 6개 지문을 계산해 각 리스트에 추가.
- `pd.DataFrame(X, columns=[f"{k}_{j:04d}" ...])` : 지문 배열을 표로, **열 이름에 종류 접두사**(ecfp4_0000 …) → 나중에 블록별로 골라 쓰기 편함.

### 셀 3 — 최종 조립 & 저장
SMILES + 6지문 + 2D desc + 3D desc + potency를 좌우로 붙여 CSV로 저장한다. (열이 ~5천 개라 Excel은 생략)

In [ ]:
# 최종 조립: canonical_smiles + [6 fingerprint] + [선택 2D desc] + [3D desc] + potency
meta = base[["canonical_smiles"]].reset_index(drop=True)
desc2d = base[sel2d].reset_index(drop=True)
# 3D는 물질 이름으로 병합(대형 분자 15개는 NaN → 학습 때 채움)
d3m = meta.merge(d3, on="canonical_smiles", how="left").drop(columns=["canonical_smiles"])
pot = base[["potency"]].reset_index(drop=True)

final = pd.concat([meta] + fp_dfs + [desc2d, d3m, pot], axis=1)
OUT = "data/HSD17B13_final_training_1to1_v2.csv"
final.to_csv(OUT, index=False)
n_fp = sum(df.shape[1] for df in fp_dfs)
print("최종 학습데이터:", final.shape,
      f"(smiles 1 + fingerprint {n_fp} + 2D desc {len(sel2d)} + 3D desc {d3m.shape[1]} + potency 1)")
print("저장:", OUT, "| potency 분포:", dict(final.potency.value_counts()))
print("열 접두사로 블록 구분: ecfp4_/rdkit_/atompair_/topotorsion_/maccs_/avalon_ (지문), d3_ (3D), 나머지=2D desc")

🔎 **코드 뜯어보기 (셀 3)**
- `meta.merge(d3, on="canonical_smiles", how="left")` : 3D descriptor를 **물질 이름 기준으로 병합**(3D 없는 대형 분자는 NaN → 학습 시 채움).
- `pd.concat([meta] + fp_dfs + [desc2d, d3m, pot], axis=1)` : 리스트를 이어 **좌우로 전부 결합**. potency는 맨 끝.
- `.to_csv(OUT, index=False)` : CSV로 저장(파일명 `_v2` — 기존 1지문 버전과 구분).